In [1]:
pip install shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip uninstall shap numpy -y

Found existing installation: shap 0.42.0
Uninstalling shap-0.42.0:
  Successfully uninstalled shap-0.42.0
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


In [3]:

!pip install "numpy>=1.24,<2"


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)


In [4]:
!pip install shap==0.42.0

  Using cached shap-0.42.0-cp312-cp312-win_amd64.whl


In [5]:
import numpy as np
import shap
import pandas as pd
print(f"✅ NumPy  : {np.__version__}")
print(f"✅ SHAP   : {shap.__version__}")
print(f"✅ Pandas : {pd.__version__}")

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


✅ NumPy  : 1.26.4
✅ SHAP   : 0.42.0
✅ Pandas : 2.2.2


In [ ]:
# ============================================
# 05_analyse_shap.ipynb - Andy
# Analyse SHAP du modèle final (CatBoost)
# Version améliorée - Cas extrêmes pour les Waterfall
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Toutes les librairies sont chargées !")


# ============================================
# 1. CHARGER LE MODÈLE ET LES DONNÉES
# ============================================

print("\n📂 Chargement du modèle et des données...")

modele_data = joblib.load('modele_scoring_credit.joblib')
modele = modele_data['modele']
features = modele_data['features']
cible = modele_data['cible']
algorithme = modele_data['algorithme']

print(f"✅ Modèle chargé : {algorithme}")
print(f"✅ {len(features)} features")

df = pd.read_csv('Loan_Default_Cameroun_Encode.csv')
print(f"✅ Dataset chargé : {df.shape}")

CIBLE = "statut_remboursement"
COLONNES_NON_FEATURES = [
    "id_client",
    "genre",
    "tranche_age",
    "niveau_education",
    "membre_tontine",
    "activite_saisonniere",
    "utilisation_mobile_money",
]

FEATURES = [c for c in df.columns if c not in COLONNES_NON_FEATURES and c != CIBLE]
X = df[FEATURES]
y = df[CIBLE]

from sklearn.model_selection import train_test_split
RANDOM_SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

print(f"✅ Train : {X_train.shape[0]} lignes")
print(f"✅ Test  : {X_test.shape[0]} lignes")
print(f"✅ Proportion de défaut dans le test : {y_test.mean():.2%}")


# ============================================
# 2. ANALYSE SHAP - CALCUL DES VALEURS
# ============================================

print("\n🔮 Calcul des valeurs SHAP...")

explainer = shap.TreeExplainer(modele)
X_sample = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

print("✅ Valeurs SHAP calculées")


# ============================================
# 3. GRAPHIQUE D'IMPORTANCE (VERSION MANUELLE)
# ============================================

print("\n📊 Génération du graphique d'importance...")

mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]
sorted_features = [features[i] for i in sorted_idx[:16]]
sorted_importance = mean_abs_shap[sorted_idx[:16]]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(sorted_importance)), sorted_importance, color='steelblue')

for i, (bar, val) in enumerate(zip(bars, sorted_importance)):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, 
            f'{val:.4f}', va='center', fontsize=9)

ax.set_yticks(range(len(sorted_importance)))
ax.set_yticklabels(sorted_features)
ax.set_xlabel('Importance moyenne absolue SHAP', fontsize=12)
ax.set_title(f'Importance des variables - {algorithme}', fontsize=14)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('shap_importance_globale.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Graphique d'importance sauvegardé")


# ============================================
# 4. GRAPHIQUE EN BARRES
# ============================================

print("\n📊 Génération du graphique en barres...")

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': mean_abs_shap
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'][:15], importance_df['Importance'][:15])
plt.xlabel('Importance moyenne absolue SHAP')
plt.title(f'Top 15 variables importantes - {algorithme}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('shap_importance_barres.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Graphique en barres sauvegardé")


# ============================================
# 5. WATERFALL PLOTS (CAS EXTRÊMES)
# ============================================

print("\n📊 Génération du waterfall plot...")

# Récupérer les vraies valeurs cibles pour l'échantillon
y_sample = y_test.loc[X_sample.index]

# Obtenir les probabilités prédites pour identifier les cas extrêmes
y_pred_proba = modele.predict_proba(X_sample)[:, 1]  # Probabilité de défaut

# Sélection du client le PLUS risqué (probabilité de défaut maximale)
idx_defaut = np.argmax(y_pred_proba)

# Sélection du client le PLUS sûr (probabilité de défaut minimale)
idx_rembourse = np.argmin(y_pred_proba)

# Vérification : s'assurer que les deux indices sont différents
if idx_defaut == idx_rembourse:
    idx_rembourse = (idx_rembourse + 1) % len(X_sample)

print(f"   → Client en défaut (probabilité = {y_pred_proba[idx_defaut]:.2%})")
print(f"   → Client ayant remboursé (probabilité = {y_pred_proba[idx_rembourse]:.2%})")

# Waterfall plot - Client en défaut
print("   → Génération du waterfall pour le client en défaut...")
plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_defaut],
        base_values=explainer.expected_value,
        data=X_sample.iloc[idx_defaut],
        feature_names=features
    ),
    show=False
)
plt.title(f'Explication du score - Client en défaut (probabilité = {y_pred_proba[idx_defaut]:.2%})', fontsize=14)
plt.tight_layout()
plt.savefig('shap_waterfall_defaut.png', dpi=300, bbox_inches='tight')
plt.show()

# Waterfall plot - Client ayant remboursé
print("   → Génération du waterfall pour le client ayant remboursé...")
plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_rembourse],
        base_values=explainer.expected_value,
        data=X_sample.iloc[idx_rembourse],
        feature_names=features
    ),
    show=False
)
plt.title(f'Explication du score - Client ayant remboursé (probabilité = {y_pred_proba[idx_rembourse]:.2%})', fontsize=14)
plt.tight_layout()
plt.savefig('shap_waterfall_rembourse.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Waterfall plots sauvegardés")


# ============================================
# 6. GRAPHIQUE DE DÉPENDANCE
# ============================================

print("\n📊 Génération du graphique de dépendance...")

top_feature = importance_df.iloc[0]['Feature']
print(f"🔍 Feature la plus importante : {top_feature}")

plt.figure(figsize=(10, 6))
shap.dependence_plot(top_feature, shap_values, X_sample, 
                     feature_names=features, show=False)
plt.title(f'Dépendance SHAP - {top_feature}', fontsize=14)
plt.tight_layout()
plt.savefig('shap_dependence_top.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Graphique de dépendance sauvegardé")


# ============================================
# 7. SYNTHÈSE
# ============================================

print("\n" + "="*70)
print("📊 SYNTHÈSE DE L'ANALYSE SHAP")
print("="*70)

print(f"\n🏆 Top 5 variables les plus importantes ({algorithme}) :")
for i, (feature, importance) in enumerate(zip(
    importance_df['Feature'][:5], 
    importance_df['Importance'][:5]
), 1):
    print(f"   {i}. {feature} : {importance:.4f}")

print(f"\n💡 Le modèle {algorithme} base ses décisions principalement sur :")
for i in range(3):
    print(f"   → {importance_df.iloc[i]['Feature']}")

print("\n📈 Résumé des graphiques produits :")
print("   • shap_importance_globale.png - Importance globale (manuel)")
print("   • shap_importance_barres.png - Top 15 variables")
print("   • shap_waterfall_defaut.png - Client en défaut (cas extrême)")
print("   • shap_waterfall_rembourse.png - Client remboursé (cas extrême)")
print("   • shap_dependence_top.png - Dépendance de la variable principale")

print("\n✅ Analyse SHAP terminée !")

✅ Toutes les librairies sont chargées !

📂 Chargement du modèle et des données...
✅ Modèle chargé : CatBoost
✅ 16 features
